In [2]:
import napari
from tifffile import imread
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import trackpy as tp

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [123]:
# Load image

image = imread('/Volumes/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Unperturbed/50ms/Denoised_merged/12_100tp_561-40-50ms-1000g_1_conf561_merged.tif')

In [ ]:
print(image.dtype, image.shape, np.min(image), np.max(image))

In [125]:
# Loads image mask

image_mask = imread('/Volumes/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Unperturbed/ROIs/Cytoplasm/12_100tp_561-40-50ms-1000g_1_conf561_merged_ROIs.tif')

In [ ]:
print(image_mask .dtype, image_mask .shape, np.min(image_mask ), np.max(image_mask ))

In [ ]:
# Initializes Napari viewer

viewer = napari.Viewer()

In [ ]:
# Adds image to Napari viewer

viewer.add_image(image)

In [ ]:
# Adds mask image to Napari viewer

viewer.add_image(image_mask,
                 colormap="gist_earth",
                 contrast_limits=[0,8],
                 opacity=0.2)

In [ ]:
# Loads detected spots

spots_path = r"/Volumes/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Unperturbed/50ms/Spot_detection/Run20241210/Factor4/12_100tp_561-40-50ms-1000g_1_conf561_merged_spots.csv"
spots = pd.read_csv(spots_path)
spots.head()

In [ ]:
# Creates df that contains xyct axes in the same order as image --> tcxy

spots_tcyxcoord = spots.iloc[:, np.r_[0,3,2]]
spots_tcyxcoord.insert(1, "channel", 0)
spots_tcyxcoord

In [ ]:
# Creates array from df (necessary to use in the points layer of Napari)

spots_tcyxarray = spots_tcyxcoord.to_numpy()
spots_tcyxarray

In [ ]:
# Adds detected spots to points layer

viewer.add_points(spots_tcyxcoord,
                  face_color='#ffffff00',
                  edge_color='magenta',
                  name='spots_log')

In [ ]:
# Loads tracks

tracks_path = r"/Volumes/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Unperturbed/50ms/Tracking/Factor4Link5/12_100tp_561-40-50ms-1000g_1_conf561_merged_spots_tracks.csv"
tracks = pd.read_csv(tracks_path)
tracks.head()

In [ ]:
# Takes columns "particle", "frame", "x", "y" and sorts into format required by napari for visualization
# Inserts channel information right before y and x coordinate, according to image dimensions

tracks_vis = tracks.iloc[:, np.r_[13,0,3,2]].sort_values(["unique_id", "frame"])
tracks_vis.insert(2, "channel", 0)
tracks_vis.head(20)

In [ ]:
viewer.add_tracks(tracks_vis[['unique_id', 'frame', 'channel', 'y', 'x']], name='tracks', 
                  properties={
                      'id': tracks_vis['unique_id'].values,
                             },
                  color_by='id',
                  colormap='hsv', 
                 )

: 

In [ ]:
# Splits up ROIs

tracks_ROIs = [j for i,j in tracks.groupby("roi_id")]
tracks_ROIs[1].head()

In [ ]:
# Plots tracks for one ROI of an image

fig, ax = plt.subplots()
tp.plot_traj(tracks_ROIs[0], ax=ax)

ax.set_xlim(320,530)
ax.set_ylim(735,690)

plt.show()